In [ ]:
import pandas as pd

BMKG_PATH = '/Volumes/Local Disk/Code_Git/S3_code/seismic/Indonesian_Earthquake_Catalog_BMKG_1998_2024/BMKG_Earthquake_Catalog.csv'
USGS_PATH = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/output/usgs_events_indonesia_polygon_buffer2deg.csv'

def deep_check(path, label):
    df = pd.read_csv(path)
    print(f"\n{'#'*10} ANALISIS KUALITAS: {label} {'#'*10}")
    
    # 1. Cek Statistik Magnitudo (Deteksi Outlier)
    # Mencari magnitudo ekstrem yang tidak masuk akal (misal > 9.5 atau < 0)
    mag_col = [c for c in df.columns if 'mag' in c.lower()][0]
    print(f"Rentang Magnitudo: {df[mag_col].min()} s/d {df[mag_col].max()}")
    
    # 2. Cek Kedalaman (Seringkali ada format yang salah atau negatif)
    depth_col = [c for c in df.columns if 'depth' in c.lower() or 'kedalaman' in c.lower()][0]
    print(f"Rentang Kedalaman (km): {df[depth_col].min()} s/d {df[depth_col].max()}")
    
    # 3. Cek Rentang Tahun (Untuk memastikan data tidak ada yang aneh)
    time_col = [c for c in df.columns if any(x in c.lower() for x in ['time', 'date', 'tanggal'])][0]
    dates = pd.to_datetime(df[time_col], errors='coerce')
    print(f"Rentang Waktu: {dates.min().year} s/d {dates.max().year}")
    
    # 4. Deteksi Duplikasi Event berdasarkan koordinat dan waktu kasar
    # Jika ada event dalam rentang 1 menit di koordinat yang sama, itu indikasi duplikat
    print(f"Data Null di kolom waktu: {dates.isnull().sum()}")

deep_check(BMKG_PATH, "BMKG")
deep_check(USGS_PATH, "USGS")

In [ ]:
import pandas as pd

BMKG_PATH = '/Volumes/Local Disk/Code_Git/S3_code/seismic/Indonesian_Earthquake_Catalog_BMKG_1998_2024/BMKG_Earthquake_Catalog.csv'
USGS_PATH = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/output/usgs_events_indonesia_polygon_buffer2deg.csv'

def check_each_column(path, label):
    df = pd.read_csv(path)
    print(f"\n{'='*10} ANALISIS KOLOM: {label} {'='*10}")
    
    # Membuat ringkasan statistik per kolom
    summary = pd.DataFrame({
        'Tipe Data': df.dtypes,
        'Null (%)': (df.isnull().sum() / len(df) * 100).round(2),
        'Unique': df.nunique()
    })
    
    print(summary)
    
    # Cek apakah ada kolom dengan nama yang berpotensi membingungkan
    print("\n--- Catatan Penting ---")
    if 'time' in df.columns:
        print("✅ Kolom 'time' ditemukan.")
    else:
        print("⚠️ Kolom 'time' tidak ditemukan (Perlu di-mapping).")

check_each_column(BMKG_PATH, "BMKG")
check_each_column(USGS_PATH, "USGS")

In [ ]:
import pandas as pd
import numpy as np

# Path File
BMKG_PATH = '/Volumes/Local Disk/Code_Git/S3_code/seismic/Indonesian_Earthquake_Catalog_BMKG_1998_2024/BMKG_Earthquake_Catalog.csv'
USGS_PATH = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/output/usgs_events_indonesia_polygon_buffer2deg.csv'

def final_audit(path, label):
    df = pd.read_csv(path)
    print(f"\n{'='*10} FINAL AUDIT: {label} {'='*10}")
    print(f"Total Baris: {len(df)}")
    
    # Memeriksa kolom-kolom kunci yang akan kita gunakan
    kunci = ['time', 'latitude', 'longitude', 'depth', 'magnitude']
    
    # Membuat laporan ringkas
    report = []
    for col in df.columns:
        null_pct = (df[col].isnull().sum() / len(df)) * 100
        report.append({
            'Kolom': col,
            'Tipe': df[col].dtype,
            'Null %': round(null_pct, 2)
        })
    
    report_df = pd.DataFrame(report)
    print(report_df.to_string(index=False))
    
    # Cek duplikat total
    print(f"\nJumlah Baris Duplikat: {df.duplicated().sum()}")

# Jalankan Audit
final_audit(BMKG_PATH, "BMKG")
final_audit(USGS_PATH, "USGS")

In [ ]:
import pandas as pd

master_df = pd.read_csv('/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/output/master_catalog_final.csv')

print("--- Label Kolom di Master Katalog ---")
print(master_df.columns.tolist())

# Cek apakah 6 label krusial sudah tersedia
labels_krusial = ['time', 'latitude', 'longitude', 'depth', 'magnitude', 'id']
missing = [l for l in labels_krusial if l not in master_df.columns]

if not missing:
    print("\n✅ SEMUA LABEL SUDAH LENGKAP & SIAP DIGUNAKAN")
else:
    print(f"\n⚠️ Label berikut HILANG: {missing}")

In [ ]:
import pandas as pd
import os

# --- Path ---
BMKG_PATH = '/Volumes/Local Disk/Code_Git/S3_code/seismic/Indonesian_Earthquake_Catalog_BMKG_1998_2024/BMKG_Earthquake_Catalog.csv'
USGS_PATH = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/output/usgs_events_indonesia_polygon_buffer2deg.csv'
OUTPUT_MASTER = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/output/master_catalog_full_detailed.csv'

def merge_full_detailed_catalog():
    # 1. Load Data
    df_bmkg = pd.read_csv(BMKG_PATH)
    df_usgs = pd.read_csv(USGS_PATH)

    # 2. Normalisasi & Penamaan Kolom BMKG ke Standar Internasional
    df_bmkg['time'] = pd.to_datetime(df_bmkg['Date'] + ' ' + df_bmkg['Time (UTC)'], utc=True)
    df_bmkg = df_bmkg.rename(columns={
        'Latitude': 'latitude', 'Longitude': 'longitude', 
        'Depth (km)': 'depth', 'Magnitude': 'magnitude', 'Event ID': 'id'
    })
    df_bmkg['origin_catalog'] = 'BMKG'

    # 3. Normalisasi USGS
    df_usgs['time'] = pd.to_datetime(df_usgs['time'], utc=True)
    df_usgs = df_usgs.rename(columns={'mag': 'magnitude'})
    df_usgs['origin_catalog'] = 'USGS'

    # 4. Penggabungan Luas (Outer Join-like Concatenation)
    # pd.concat akan menjaga kolom unik dari kedua sumber
    master_full = pd.concat([df_bmkg, df_usgs], ignore_index=True, sort=False)

    # 5. Sorting
    master_full = master_full.sort_values('time').reset_index(drop=True)

    # 6. Simpan
    master_full.to_csv(OUTPUT_MASTER, index=False)
    print(f"✅ Katalog Master Lengkap Terbentuk!")
    print(f"Total baris: {len(master_full)}")
    print(f"Total kolom (atribut): {len(master_full.columns)}")
    print(f"Lokasi: {OUTPUT_MASTER}")

if __name__ == "__main__":
    merge_full_detailed_catalog()

In [ ]:
import pandas as pd
import os

# --- KONFIGURASI PATH ---
BMKG_PATH = '/Volumes/Local Disk/Code_Git/S3_code/seismic/Indonesian_Earthquake_Catalog_BMKG_1998_2024/BMKG_Earthquake_Catalog.csv'
USGS_PATH = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/output/usgs_events_indonesia_polygon_buffer2deg.csv'
OUTPUT_FILE = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/output/master_catalog_final.csv'

def main():
    print("🚀 Memulai proses penggabungan katalog master...")

    # 1. LOAD DATA
    df_bmkg = pd.read_csv(BMKG_PATH)
    df_usgs = pd.read_csv(USGS_PATH)

    # 2. NORMALISASI BMKG
    # Gabungkan Date dan Time, lalu konversi ke UTC
    df_bmkg['time'] = pd.to_datetime(df_bmkg['Date'] + ' ' + df_bmkg['Time (UTC)'], utc=True)
    df_bmkg = df_bmkg.rename(columns={
        'Latitude': 'latitude', 
        'Longitude': 'longitude', 
        'Depth (km)': 'depth', 
        'Magnitude': 'magnitude', 
        'Event ID': 'id'
    })
    df_bmkg['origin'] = 'BMKG'

    # 3. NORMALISASI USGS
    # Gunakan format='mixed' untuk menangani variasi string waktu
    df_usgs['time'] = pd.to_datetime(df_usgs['time'], utc=True, format='mixed')
    df_usgs = df_usgs.rename(columns={'mag': 'magnitude'})
    df_usgs['origin'] = 'USGS'

    # 4. PENGGABUNGAN (CONCATENATION)
    # Pilih kolom kunci saja untuk menjaga konsistensi
    kunci = ['time', 'latitude', 'longitude', 'depth', 'magnitude', 'id', 'origin']
    master_df = pd.concat([df_bmkg[kunci], df_usgs[kunci]], ignore_index=True)

    # 5. DEDUPLIKASI & SORTING
    # Mengurutkan berdasarkan waktu terlebih dahulu
    master_df = master_df.sort_values('time')
    
    # Hapus duplikat berdasarkan waktu dan lokasi (toleransi presisi floating point)
    master_df = master_df.drop_duplicates(subset=['time', 'latitude', 'longitude'], keep='first')
    
    # Reset index agar rapi
    master_df = master_df.reset_index(drop=True)

    # 6. SIMPAN HASIL
    os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)
    master_df.to_csv(OUTPUT_FILE, index=False)
    
    print(f"\n✅ Katalog Master Berhasil Dibuat!")
    print(f"Total baris unik: {len(master_df)}")
    print(f"Lokasi file: {OUTPUT_FILE}")

if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd
import os

# --- KONFIGURASI PATH ---
BMKG_PATH = '/Volumes/Local Disk/Code_Git/S3_code/seismic/Indonesian_Earthquake_Catalog_BMKG_1998_2024/BMKG_Earthquake_Catalog.csv'
USGS_PATH = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/output/usgs_events_indonesia_polygon_buffer2deg.csv'
OUTPUT_FILE = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/output/master_catalog_final.csv'

def main():
    print("🚀 Memulai proses penggabungan katalog master...")

    # 1. LOAD DATA
    df_bmkg = pd.read_csv(BMKG_PATH)
    df_usgs = pd.read_csv(USGS_PATH)

    # 2. NORMALISASI BMKG
    # Gabungkan Date dan Time, lalu konversi ke UTC
    df_bmkg['time'] = pd.to_datetime(df_bmkg['Date'] + ' ' + df_bmkg['Time (UTC)'], utc=True)
    df_bmkg = df_bmkg.rename(columns={
        'Latitude': 'latitude', 
        'Longitude': 'longitude', 
        'Depth (km)': 'depth', 
        'Magnitude': 'magnitude', 
        'Event ID': 'id'
    })
    df_bmkg['origin'] = 'BMKG'

    # 3. NORMALISASI USGS
    # Gunakan format='mixed' untuk menangani variasi string waktu
    df_usgs['time'] = pd.to_datetime(df_usgs['time'], utc=True, format='mixed')
    df_usgs = df_usgs.rename(columns={'mag': 'magnitude'})
    df_usgs['origin'] = 'USGS'

    # 4. PENGGABUNGAN (CONCATENATION)
    # Pilih kolom kunci saja untuk menjaga konsistensi
    kunci = ['time', 'latitude', 'longitude', 'depth', 'magnitude', 'id', 'origin']
    master_df = pd.concat([df_bmkg[kunci], df_usgs[kunci]], ignore_index=True)

    # 5. DEDUPLIKASI & SORTING
    # Mengurutkan berdasarkan waktu terlebih dahulu
    master_df = master_df.sort_values('time')
    
    # Hapus duplikat berdasarkan waktu dan lokasi (toleransi presisi floating point)
    master_df = master_df.drop_duplicates(subset=['time', 'latitude', 'longitude'], keep='first')
    
    # Reset index agar rapi
    master_df = master_df.reset_index(drop=True)

    # 6. SIMPAN HASIL
    os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)
    master_df.to_csv(OUTPUT_FILE, index=False)
    
    print(f"\n✅ Katalog Master Berhasil Dibuat!")
    print(f"Total baris unik: {len(master_df)}")
    print(f"Lokasi file: {OUTPUT_FILE}")

if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd

# Path ke file master katalog hasil penggabungan
MASTER_PATH = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/output/master_catalog_final.csv'

def check_catalog_composition():
    # Membaca file master
    master_df = pd.read_csv(MASTER_PATH)
    
    # Menghitung jumlah unik berdasarkan kolom 'origin'
    counts = master_df['origin'].value_counts()
    
    print("--- Komposisi Data dalam Master Katalog ---")
    print(counts)
    print("-" * 40)
    print(f"Total Baris Keseluruhan: {len(master_df)}")

if __name__ == "__main__":
    check_catalog_composition()

In [ ]:
import pandas as pd

# Path ke file master
MASTER_PATH = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/output/master_catalog_final.csv'

def check_magnitude_range():
    master_df = pd.read_csv(MASTER_PATH)
    
    # Kelompokkan berdasarkan 'origin' dan ambil statistik kolom 'magnitude'
    stats = master_df.groupby('origin')['magnitude'].describe()
    
    print("--- Rentang Magnitudo (Min, Max, Mean) ---")
    # Menampilkan kolom min, max, mean, dan count
    print(stats[['min', 'max', 'mean', 'count']])

if __name__ == "__main__":
    check_magnitude_range()

In [29]:
import pandas as pd

# Path ke file master
MASTER_PATH = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/output/master_catalog_final.csv'

def check_columns():
    # Membaca data
    df = pd.read_csv(MASTER_PATH)
    
    print("--- Daftar Nama Kolom dalam Master Katalog ---")
    for i, col in enumerate(df.columns):
        print(f"{i+1}. {col}")
        
    print("\n--- Sampel Data (5 baris pertama) ---")
    print(df.head())

if __name__ == "__main__":
    check_columns()

--- Daftar Nama Kolom dalam Master Katalog ---
1. time
2. latitude
3. longitude
4. depth
5. magnitude
6. id
7. origin

--- Sampel Data (5 baris pertama) ---
                        time  latitude  longitude  depth  magnitude  \
0  1998-01-09 09:45:43+00:00     -5.39     126.17  390.0        4.1   
1  1998-01-12 08:05:50+00:00     -3.04     128.36  100.0        4.4   
2  1998-01-18 12:03:17+00:00     -2.65     132.13   32.0        4.8   
3  1998-01-29 15:05:24+00:00      6.76     127.28   33.0        3.8   
4  1998-02-05 06:15:24+00:00      6.88     127.22   33.0        4.3   

                        id origin  
0  BMKG-19980109094543-001   BMKG  
1  BMKG-19980112080550-001   BMKG  
2  BMKG-19980118120317-001   BMKG  
3  BMKG-19980129150524-001   BMKG  
4  BMKG-19980205061524-001   BMKG  


In [31]:
from obspy.clients.fdsn import Client
import os
import time

# 1. Definisi Direktori dan Client
OUTPUT_DIR = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Client untuk berbagai sumber
clients = {
    "BMKG": Client("https://geof.bmkg.go.id"),
    "EarthScope": Client("https://service.iris.edu")
}

# 2. Target yang akan diambil
# Catatan: Network IA dan VG adalah spesifik BMKG. 
# Kita akan cari network riset global (seperti II, IU) melalui EarthScope.
targets = [
    {"source": "BMKG", "network": "IA"},
    {"source": "BMKG", "network": "VG"},
    {"source": "EarthScope", "network": "II"}, # Global Seismographic Network
    {"source": "EarthScope", "network": "IU"}  # Global Seismographic Network
]

for t in targets:
    net = t['network']
    src = t['source']
    try:
        print(f"Mengakses {src} untuk network: {net}...")
        inventory = clients[src].get_stations(
            network=net, 
            minlatitude=-11.0, maxlatitude=6.0, 
            minlongitude=95.0, maxlongitude=141.0,
            level="station"
        )
        
        file_path = os.path.join(OUTPUT_DIR, f"inventory_{src}_{net}.xml")
        inventory.write(file_path, format="STATIONXML")
        
        print(f"✅ Berhasil menyimpan metadata dari {src} ({net})")
        print(f"💾 Lokasi: {file_path}")
        
    except Exception as e:
        print(f"❌ Gagal mengambil {net} dari {src}: {e}")
    
    time.sleep(2)

Mengakses BMKG untuk network: IA...
✅ Berhasil menyimpan metadata dari BMKG (IA)
💾 Lokasi: /Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/output/inventory_BMKG_IA.xml
Mengakses BMKG untuk network: VG...
✅ Berhasil menyimpan metadata dari BMKG (VG)
💾 Lokasi: /Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/output/inventory_BMKG_VG.xml
Mengakses EarthScope untuk network: II...
✅ Berhasil menyimpan metadata dari EarthScope (II)
💾 Lokasi: /Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/output/inventory_EarthScope_II.xml
Mengakses EarthScope untuk network: IU...
❌ Gagal mengambil IU dari EarthScope: No data available for request.
HTTP Status code: 204
Detailed response of server:




In [ ]:
# Contoh memfilter stasiun hanya dari jaringan prioritas
target_networks = ["IA", "VG"] 

for net in target_networks:
    inventory = client.get_stations(network=net, 
                                    minlatitude=-11.0, maxlatitude=6.0, 
                                    minlongitude=95.0, maxlongitude=141.0,
                                    level="station")
    print(f"Berhasil menarik data stasiun untuk network: {net}")

In [32]:
from obspy import read_inventory
import os

OUTPUT_DIR = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/output'
files_to_merge = ['inventory_BMKG_IA.xml', 'inventory_BMKG_VG.xml', 'inventory_EarthScope_II.xml']

# Inisialisasi inventory kosong dari file pertama
master_inv = read_inventory(os.path.join(OUTPUT_DIR, files_to_merge[0]))

# Gabungkan dengan file lainnya
for f in files_to_merge[1:]:
    inv = read_inventory(os.path.join(OUTPUT_DIR, f))
    master_inv += inv

# Simpan sebagai master inventory
master_inv.write(os.path.join(OUTPUT_DIR, 'master_station_inventory.xml'), format="STATIONXML")
print("Berhasil menggabungkan semua stasiun ke: master_station_inventory.xml")

Berhasil menggabungkan semua stasiun ke: master_station_inventory.xml


In [41]:
from obspy import read_inventory
from obspy.geodetics import locations2degrees
import pandas as pd
import os

# Definisi direktori
OUTPUT_DIR = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/output'
INVENTORY_PATH = os.path.join(OUTPUT_DIR, 'master_station_inventory.xml')
DATASET_PATH = os.path.join(OUTPUT_DIR, 'balanced_dataset.csv')

# 1. Muat Master Inventory
master_inv = read_inventory(INVENTORY_PATH)

# 2. Muat Dataset Gempa (sekarang file sudah ada!)
df = pd.read_csv(DATASET_PATH)

# 3. Fungsi pencari stasiun terdekat
def get_nearest_station(lat, lon, max_dist=1.5):
    best_st = None
    min_dist = max_dist
    
    for network in master_inv:
        for station in network:
            dist = locations2degrees(lat, lon, station.latitude, station.longitude)
            if dist < min_dist:
                min_dist = dist
                best_st = station
    return best_st

# 4. Uji coba pada 5 gempa pertama
print(f"Mencari stasiun untuk 5 gempa pertama...")
for i in range(5):
    sample = df.iloc[i]
    station = get_nearest_station(sample['latitude'], sample['longitude'])
    if station:
        print(f"Gempa ID {sample['id']} -> Stasiun terdekat: {station.code} (Jarak: ~{locations2degrees(sample['latitude'], sample['longitude'], station.latitude, station.longitude):.2f} derajat)")
    else:
        print(f"Gempa ID {sample['id']} -> Tidak ada stasiun dalam radius 1.5 derajat.")

Mencari stasiun untuk 5 gempa pertama...
Gempa ID BMKG-20170507103900-001 -> Tidak ada stasiun dalam radius 1.5 derajat.
Gempa ID BMKG-20221217172936-001 -> Tidak ada stasiun dalam radius 1.5 derajat.
Gempa ID BMKG-20230929052358-001 -> Tidak ada stasiun dalam radius 1.5 derajat.
Gempa ID BMKG-20171014061303-001 -> Tidak ada stasiun dalam radius 1.5 derajat.
Gempa ID BMKG-20120217215023-001 -> Tidak ada stasiun dalam radius 1.5 derajat.


In [39]:
import os

# 1. Definisikan jalur yang kita tuju
OUTPUT_DIR = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/output'
TARGET_FILE = os.path.join(OUTPUT_DIR, 'balanced_dataset.csv')

# 2. Cek apakah path itu sendiri adalah file, direktori, atau tidak ada
print(f"Path target: {TARGET_FILE}")
print(f"Apakah path ini ada? {os.path.exists(TARGET_FILE)}")
print(f"Apakah path ini adalah file? {os.path.isfile(TARGET_FILE)}")

# 3. Mari lihat list direktori lagi dengan lebih detail
print("\nIsi direktori output (dengan pengecekan nama):")
files = os.listdir(OUTPUT_DIR)
for f in files:
    print(f"- {f}")
    if f.lower().strip() == 'balanced_dataset.csv':
        print("  ^^^ DITEMUKAN!")

Path target: /Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/output/balanced_dataset.csv
Apakah path ini ada? False
Apakah path ini adalah file? False

Isi direktori output (dengan pengecekan nama):
- katalog_gabungan_usgs_bmkg.csv
- master_catalog_full_detailed.csv
- pembuatan_katalog_gabungan.ipynb
- .DS_Store
- usgs_events_indonesia_polygon_buffer2deg.csv
- inventory_VG.xml
- master_catalog_final.csv
- katalog_gabungan_final_mcu.csv
- catalog_usgs_bmkg_irisan.csv
- log_success_dynamic_2003_2005.csv
- inventory_IA.xml
- katalog_irisan.csv
- usgs_events_indonesia.csv
- log_success_dynamic
- inventory_EarthScope_II.xml
- inventory_BMKG_IA.xml
- log_success_dynamic_2000_2001.csv
- waveform_buffer
- master_station_inventory.xml
- usgs_events_indonesia_polygon.csv
- katalog_tanpa_irisan.csv
- log_success_dynamic_2021_2021.csv
- inventory_BMKG_VG.xml


In [40]:
import pandas as pd
import os

# 1. Definisi path yang sudah kita verifikasi
OUTPUT_DIR = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/output'
MASTER_FILE = os.path.join(OUTPUT_DIR, 'master_catalog_final.csv')
TARGET_FILE = os.path.join(OUTPUT_DIR, 'balanced_dataset.csv')

# 2. Baca file master
print(f"Membaca katalog dari: {MASTER_FILE}")
master_df = pd.read_csv(MASTER_FILE)

# 3. Stratifikasi (sesuai rencana)
master_df['mag_bin'] = pd.cut(master_df['magnitude'], bins=[3.0, 3.5, 4.0, 4.5, 5.0, 5.5, 6.0, 9.0])
balanced_df = master_df.groupby('mag_bin', group_keys=False, observed=True).apply(
    lambda x: x.sample(min(len(x), 1000))
)

# 4. Simpan file
balanced_df.to_csv(TARGET_FILE, index=False)
print(f"✅ File berhasil dibuat di: {TARGET_FILE}")
print(f"Jumlah baris: {len(balanced_df)}")

Membaca katalog dari: /Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/output/master_catalog_final.csv
✅ File berhasil dibuat di: /Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/output/balanced_dataset.csv
Jumlah baris: 6543


/var/folders/lt/2mkl6ry53ll9fdk2br6skfgw0000gn/T/ipykernel_98574/1961349909.py:15: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  balanced_df = master_df.groupby('mag_bin', group_keys=False, observed=True).apply(


In [43]:
from obspy import read_inventory
from obspy.geodetics import locations2degrees
import pandas as pd
import os

# 1. Definisi Path
OUTPUT_DIR = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/output'
INVENTORY_PATH = os.path.join(OUTPUT_DIR, 'master_station_inventory.xml')
DATASET_PATH = os.path.join(OUTPUT_DIR, 'balanced_dataset.csv')
OUTPUT_PATH = os.path.join(OUTPUT_DIR, 'dataset_with_stations.csv')

# 2. Muat data
master_inv = read_inventory(INVENTORY_PATH)
df = pd.read_csv(DATASET_PATH)

# Inisialisasi kolom jika belum ada
df['nearest_station'] = None
df['network_code'] = None

# 3. Fungsi Pencari
def get_best_station_flexible(lat, lon):
    for radius in [1.5, 2.5, 4.0]:
        best_st = None
        min_dist = radius
        for network in master_inv:
            for station in network:
                dist = locations2degrees(lat, lon, station.latitude, station.longitude)
                if dist < min_dist:
                    min_dist = dist
                    best_st = station
        if best_st:
            return best_st.code, best_st.network_code
    return None, None

# 4. Pengisian DataFrame secara langsung
print("Memproses pencarian stasiun...")
for i in range(len(df)):
    st_code, net_code = get_best_station_flexible(df.loc[i, 'latitude'], df.loc[i, 'longitude'])
    df.at[i, 'nearest_station'] = st_code
    df.at[i, 'network_code'] = net_code
    
    if i % 1000 == 0:
        print(f"Progress: {i}/{len(df)}...")

# 5. Simpan
df.to_csv(OUTPUT_PATH, index=False)
print(f"✅ Selesai! Cek jumlah stasiun terisi: {df['nearest_station'].notna().sum()}")

Memproses pencarian stasiun...
Progress: 0/6543...
Progress: 1000/6543...
Progress: 2000/6543...
Progress: 3000/6543...
Progress: 4000/6543...
Progress: 5000/6543...
Progress: 6000/6543...
✅ Selesai! Cek jumlah stasiun terisi: 0


In [44]:
from obspy import read_inventory
import os

OUTPUT_DIR = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/output'
INVENTORY_PATH = os.path.join(OUTPUT_DIR, 'master_station_inventory.xml')

# Muat dan periksa
master_inv = read_inventory(INVENTORY_PATH)

print(f"Jumlah network dalam XML: {len(master_inv)}")
total_st = 0
for net in master_inv:
    print(f"Network {net.code} memiliki {len(net)} stasiun.")
    total_st += len(net)

print(f"Total stasiun terbaca: {total_st}")

Jumlah network dalam XML: 4
Network IA memiliki 5 stasiun.
Network IA memiliki 1350 stasiun.
Network VG memiliki 2 stasiun.
Network II memiliki 1 stasiun.
Total stasiun terbaca: 1358
